[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/079.%20The%20Freight%20Carrier%20Selection%20%26%20Bidding%20Problem/P79-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 79. The Freight Carrier Selection & Bidding Problem
## Tier 3: Simulated Annealing

### Purpose and scope
This notebook applies **Simulated Annealing (SA)** to assign a carrier to each freight lane. It uses a weighted objective that rewards lower bid costs and higher reliability and service scores, subject to carrier capacity limits. The method starts from a greedy assignment, explores neighboring assignments, and probabilistically accepts some worse moves while the temperature decreases.

The inputs are the lane demands, carrier bids, reliability and service scores, carrier capacities, objective weights, and SA parameters. The outputs are the best feasible assignment encountered, its weighted objective score and actual freight spend, and diagnostics describing the search. SA is a stochastic heuristic: it does not certify that the best assignment it finds is globally optimal.

### Model assumptions
- Every lane must be assigned to exactly one carrier.
- A carrier's load is the sum of the demands of its assigned lanes; it cannot exceed that carrier's capacity.
- A lane's bid is a cost per demand unit, so its freight spend is bid price multiplied by lane demand.
- Reliability is a carrier-level score, and service is a carrier-lane score. Their contributions are averaged across assigned lanes.
- The weighted objective is a score, not a dollar amount. Actual freight spend is reported separately.
- The example is deterministic when run from top to bottom with the supplied random seeds, although computation time can vary across machines.

The model does not include additional real-world factors such as contract terms, emissions, uncertainty in demand, or interactions between lanes beyond the carrier capacity limits. These are outside the scope of this example.

### Search outline
1. Construct a feasible initial assignment with a greedy rule.
2. Propose a neighbor by changing the carrier assigned to one randomly selected lane.
3. Reject a neighbor that violates a capacity limit.
4. Accept an improving feasible neighbor; accept a worse feasible neighbor with a temperature-dependent probability.
5. Reduce the temperature and retain the best feasible assignment encountered.

The initial temperature is 1000, the cooling rate is 0.995, and the iteration limit is 5000. The stopping rule also ends the search when the temperature falls to 0.01 or below. These parameters govern the search; they do not guarantee a particular solution or convergence point.

## Mathematical model and search method

Let $i \in \{1,\ldots,m\}$ index carriers and $j \in \{1,\ldots,n\}$ index lanes. Define the binary decision variable $x_{ij}$ to be 1 when carrier $i$ serves lane $j$, and 0 otherwise. Every lane is assigned exactly once, and carrier capacity is respected:

$$
\sum_{i=1}^{m} x_{ij} = 1 \quad (j=1,\ldots,n),
\qquad
\sum_{j=1}^{n} d_j x_{ij} \leq q_i \quad (i=1,\ldots,m),
\qquad
x_{ij} \in \{0,1\}.
$$

Here, $d_j$ is the demand on lane $j$ and $q_i$ is the capacity of carrier $i$. In the code, an assignment vector stores the selected carrier index for each lane; `-1` is reserved to represent an unassigned lane and is infeasible. `is_feasible` checks the assignment and capacity constraints.

For a complete assignment, let $C$ be total freight spend, $R$ be average carrier reliability, and $S$ be average lane service score. The code minimizes the following weighted score:

$$
F = \alpha C - \beta (100 R) - \gamma (10 S),
\qquad
C = \sum_{j=1}^{n} p_{a_j j} d_j,
\qquad
R = \frac{1}{n}\sum_{j=1}^{n} r_{a_j},
\qquad
S = \frac{1}{n}\sum_{j=1}^{n} s_{a_j j}.
$$

In these expressions, $a_j$ is the carrier assigned to lane $j$, $p_{ij}$ is the bid price per demand unit, $r_i$ is carrier reliability, and $s_{ij}$ is the service score for carrier $i$ on lane $j$. The parameters $\alpha$, $\beta$, and $\gamma$ are the cost, reliability, and service weights. The factors 100 and 10 are the score scalings used by this example. Because these terms combine different units and scalings, $F$ is an optimization score, whereas $C$ is the freight spend in dollars. The code counterparts are `calculate_objective` and `calculate_total_cost`.

### Simulated Annealing acceptance rule
A state is a complete feasible assignment. A neighbor changes one lane's carrier. If the neighbor has objective $F_{\mathrm{new}}$ and the current state has objective $F_{\mathrm{current}}$, define $\Delta = F_{\mathrm{new}} - F_{\mathrm{current}}$. The acceptance probability is

$$
P(\text{accept}) =
\begin{cases}
1, & \Delta < 0, \\
\exp(-\Delta/T), & \Delta \geq 0,
\end{cases}
\qquad
T_{k+1} = \rho T_k.
$$

Here, $T$ is the temperature, $k$ is the iteration number, and $\rho$ is the cooling rate. At higher temperatures, a worse move can be accepted with a greater probability; as temperature falls, the search increasingly favors improvement. This controlled acceptance helps explore assignments that a purely improving search might not reach. `simulated_annealing` implements this rule, records the search history, and separately retains the best assignment seen.

With $m$ carriers and $n$ lanes, there are at most $m^n$ complete assignments before capacity constraints are applied. Each iteration in this implementation evaluates a single-lane change and checks feasibility and objective value in time proportional to the number of lanes. The method therefore limits the work performed, but its result depends on the initial assignment, random choices, and temperature schedule.

In [ ]:
import random
import time as time_module

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the stochastic search reproducible when cells run in order.
np.random.seed(42)
random.seed(42)

## Problem data

The bid-price matrix has one row per carrier and one column per lane. Reliability is specified per carrier, while service scores are specified for each carrier-lane pair. Demand and capacity use the same units, so the capacity check compares like quantities.

In [ ]:
carriers = ["Carrier 1", "Carrier 2", "Carrier 3"]
lanes = ["Lane 1 (NYC-CHI)", "Lane 2 (LAX-DAL)"]

bid_prices = np.array([
    [500, 450],
    [480, 470],
    [520, 440],
])

reliability_scores = np.array([95, 88, 92])

service_scores = np.array([
    [90, 85],
    [88, 92],
    [87, 90],
])

demand = np.array([100, 80])
capacity_limits = np.array([150, 120, 200])

alpha = 0.6
beta = 0.3
gamma = 0.1

problem = {
    "carriers": carriers,
    "lanes": lanes,
    "bid_prices": bid_prices,
    "reliability_scores": reliability_scores,
    "service_scores": service_scores,
    "demand": demand,
    "capacity_limits": capacity_limits,
    "alpha": alpha,
    "beta": beta,
    "gamma": gamma,
}

assert bid_prices.shape == (len(carriers), len(lanes))
assert service_scores.shape == (len(carriers), len(lanes))
assert reliability_scores.shape == (len(carriers),)
assert demand.shape == (len(lanes),)
assert capacity_limits.shape == (len(carriers),)
assert np.all(demand >= 0)
assert np.all(capacity_limits >= 0)

print("Problem data:")
print(f"Carriers: {carriers}")
print(f"Lanes: {lanes}")
print(f"Demand: {demand}")
print(f"Weight parameters (alpha, beta, gamma): ({alpha}, {beta}, {gamma})")

## Objective, feasibility, and neighborhood functions

The objective function converts cost, reliability, and service into the single score used to compare assignments. Feasibility is a separate test: an assignment is eligible for the search only when every lane has a valid carrier and no carrier exceeds capacity. The neighborhood changes one lane at a time, preserving the search structure of a local stochastic search.

In [ ]:
def calculate_objective(assignments, problem):
    """Return the weighted objective score for a complete assignment."""
    demand = problem["demand"]
    bid_prices = problem["bid_prices"]
    reliability_scores = problem["reliability_scores"]
    service_scores = problem["service_scores"]

    total_cost = 0.0
    total_reliability = 0.0
    total_service = 0.0

    for lane_idx, carrier_idx in enumerate(assignments):
        total_cost += bid_prices[carrier_idx, lane_idx] * demand[lane_idx]
        total_reliability += reliability_scores[carrier_idx]
        total_service += service_scores[carrier_idx, lane_idx]

    lane_count = len(assignments)
    average_reliability = total_reliability / lane_count
    average_service = total_service / lane_count

    return (
        problem["alpha"] * total_cost
        - problem["beta"] * average_reliability * 100
        - problem["gamma"] * average_service * 10
    )


def calculate_total_cost(assignments, problem):
    """Return actual freight spend in dollars for a complete assignment."""
    total_cost = 0.0
    for lane_idx, carrier_idx in enumerate(assignments):
        total_cost += (
            problem["bid_prices"][carrier_idx, lane_idx]
            * problem["demand"][lane_idx]
        )
    return total_cost


def is_feasible(assignments, problem):
    """Check complete lane coverage and carrier capacity limits."""
    carrier_count = len(problem["carriers"])
    lane_count = len(problem["lanes"])

    if len(assignments) != lane_count:
        return False, np.zeros(carrier_count)

    carrier_utilization = np.zeros(carrier_count, dtype=float)

    for lane_idx, carrier_idx in enumerate(assignments):
        if not isinstance(carrier_idx, (int, np.integer)):
            return False, carrier_utilization
        if carrier_idx < 0 or carrier_idx >= carrier_count:
            return False, carrier_utilization
        carrier_utilization[carrier_idx] += problem["demand"][lane_idx]

    feasible = np.all(carrier_utilization <= problem["capacity_limits"])
    return bool(feasible), carrier_utilization


def generate_neighbor(current_assignments, carrier_count):
    """Change the carrier assigned to one randomly selected lane."""
    neighbor = current_assignments.copy()
    lane_idx = random.randrange(len(neighbor))
    current_carrier = neighbor[lane_idx]
    available_carriers = [
        carrier_idx
        for carrier_idx in range(carrier_count)
        if carrier_idx != current_carrier
    ]
    neighbor[lane_idx] = random.choice(available_carriers)
    return neighbor

## Greedy initialization and Simulated Annealing

The greedy rule processes lanes from highest to lowest demand and assigns each lane to the feasible carrier with the highest local score. This provides a feasible starting point; it is not a proof of global optimality. SA then explores neighboring assignments. Infeasible proposals are rejected, and each iteration advances the temperature schedule. Keeping the best assignment separately from the current assignment ensures that accepting a worse move does not discard the best state encountered.

In [ ]:
def greedy_initial_solution(problem):
    """Create a feasible initial assignment using a greedy lane order."""
    lanes = problem["lanes"]
    carriers = problem["carriers"]
    demand = problem["demand"]
    bid_prices = problem["bid_prices"]
    reliability_scores = problem["reliability_scores"]
    service_scores = problem["service_scores"]
    capacity_limits = problem["capacity_limits"]

    assignments = [-1] * len(lanes)
    carrier_utilization = np.zeros(len(carriers), dtype=float)
    lane_order = np.argsort(-demand)

    for lane_idx in lane_order:
        best_carrier = -1
        best_score = -np.inf

        for carrier_idx in range(len(carriers)):
            if (
                carrier_utilization[carrier_idx] + demand[lane_idx]
                <= capacity_limits[carrier_idx]
            ):
                cost = bid_prices[carrier_idx, lane_idx]
                reliability = reliability_scores[carrier_idx]
                service = service_scores[carrier_idx, lane_idx]
                score = (
                    problem["alpha"] * (1 / cost) * 1000
                    + problem["beta"] * reliability / 100
                    + problem["gamma"] * service / 100
                )

                if score > best_score:
                    best_score = score
                    best_carrier = carrier_idx

        if best_carrier == -1:
            raise ValueError("The greedy rule could not assign every lane feasibly.")

        assignments[lane_idx] = best_carrier
        carrier_utilization[best_carrier] += demand[lane_idx]

    feasible, _ = is_feasible(assignments, problem)
    if not feasible:
        raise AssertionError("The greedy initializer must return a feasible assignment.")
    return assignments


def simulated_annealing(
    initial_assignments,
    problem,
    initial_temp=1000,
    cooling_rate=0.995,
    max_iterations=5000,
):
    """Search for a low-objective feasible assignment using Simulated Annealing."""
    feasible, _ = is_feasible(initial_assignments, problem)
    if not feasible:
        raise ValueError("The initial assignment must be feasible.")
    if initial_temp <= 0:
        raise ValueError("Initial temperature must be positive.")
    if not 0 < cooling_rate < 1:
        raise ValueError("Cooling rate must be between zero and one.")
    if max_iterations < 0:
        raise ValueError("Maximum iterations cannot be negative.")

    start_time = time_module.time()
    current_assignments = initial_assignments.copy()
    current_obj = calculate_objective(current_assignments, problem)
    best_assignments = current_assignments.copy()
    best_obj = current_obj

    temperature = float(initial_temp)
    iteration = 0
    progress_history = []
    temperature_history = []
    acceptance_history = []

    print("Starting Simulated Annealing...")
    print(f"Initial temperature: {initial_temp}")
    print(f"Cooling rate: {cooling_rate}")
    print(f"Maximum iterations: {max_iterations}")

    while iteration < max_iterations and temperature > 0.01:
        temperature_history.append(temperature)
        neighbor_assignments = generate_neighbor(
            current_assignments,
            len(problem["carriers"]),
        )
        neighbor_feasible, _ = is_feasible(neighbor_assignments, problem)
        accept = False

        if neighbor_feasible:
            neighbor_obj = calculate_objective(neighbor_assignments, problem)
            delta = neighbor_obj - current_obj

            if delta < 0:
                accept = True
            else:
                acceptance_probability = np.exp(-delta / temperature)
                accept = random.random() < acceptance_probability

            if accept:
                current_assignments = neighbor_assignments
                current_obj = neighbor_obj

                if current_obj < best_obj:
                    best_assignments = current_assignments.copy()
                    best_obj = current_obj

        acceptance_history.append(int(accept))
        progress_history.append(current_obj)
        temperature *= cooling_rate
        iteration += 1

        if iteration % 1000 == 0:
            print(
                f"Iteration {iteration}: "
                f"Best objective = {best_obj:.2f}, "
                f"Temperature = {temperature:.4f}"
            )

    computation_time = time_module.time() - start_time
    best_feasible, _ = is_feasible(best_assignments, problem)
    if not best_feasible:
        raise AssertionError("The best recorded assignment must be feasible.")

    return {
        "best_assignments": best_assignments,
        "best_objective": best_obj,
        "final_cost": calculate_total_cost(best_assignments, problem),
        "iterations": iteration,
        "computation_time": computation_time,
        "final_temperature": temperature,
        "progress_history": progress_history,
        "temperature_history": temperature_history,
        "acceptance_history": acceptance_history,
    }


initial_solution = greedy_initial_solution(problem)
greedy_cost = calculate_total_cost(initial_solution, problem)
greedy_objective = calculate_objective(initial_solution, problem)
sa_result = simulated_annealing(initial_solution, problem)

print("\nSimulated Annealing results:")
for lane_idx in np.argsort(-problem["demand"]):
    carrier_idx = sa_result["best_assignments"][lane_idx]
    print(f"{problem['lanes'][lane_idx]}: {problem['carriers'][carrier_idx]}")

print(f"\nWeighted objective score: {sa_result['best_objective']:.2f}")
print(f"Total freight spend: ${sa_result['final_cost']:,.2f}")
print(f"Iterations: {sa_result['iterations']}")
print(f"Computation time: {sa_result['computation_time']:.3f} seconds")

## Search diagnostics and cost comparison

The first plots show the current objective score, the temperature schedule, and the rolling acceptance rate. The objective history can move upward temporarily because SA may accept worse states; the best assignment is tracked separately. The final plot compares actual freight spend for the greedy starting assignment and the best assignment found by SA. A lower bar means lower freight spend, but freight spend alone does not capture the reliability and service terms in the weighted objective.

In [ ]:
progress = sa_result["progress_history"]
temperatures = sa_result["temperature_history"]
acceptances = sa_result["acceptance_history"]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Simulated Annealing Search Diagnostics", fontsize=16, fontweight="bold")

iterations = np.arange(1, len(progress) + 1)
ax1 = axes[0, 0]
ax1.plot(iterations, progress, color="steelblue", alpha=0.7, linewidth=1)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Weighted objective score")
ax1.set_title("Current-state objective during the search")
ax1.grid(True, alpha=0.3)
window_size = min(100, max(1, len(progress) // 10))
if len(progress) >= window_size and window_size > 1:
    moving_average = pd.Series(progress).rolling(window=window_size).mean()
    ax1.plot(
        iterations,
        moving_average,
        color="darkorange",
        linewidth=2,
        label=f"Moving average (window={window_size})",
    )
    ax1.legend()

ax2 = axes[0, 1]
ax2.plot(iterations, temperatures, color="darkorange", linewidth=1)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Temperature")
ax2.set_title("Temperature schedule")
ax2.set_yscale("log")
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
acceptance_window = min(500, max(1, len(acceptances) // 5))
if acceptances:
    rolling_acceptance = pd.Series(acceptances).rolling(
        window=acceptance_window,
        min_periods=1,
    ).mean()
    ax3.plot(
        np.arange(1, len(rolling_acceptance) + 1),
        rolling_acceptance,
        color="seagreen",
        linewidth=1,
    )
ax3.set_xlabel("Iteration")
ax3.set_ylabel("Acceptance rate")
ax3.set_title(f"Rolling acceptance rate (window={acceptance_window})")
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
method_names = ["Greedy initial assignment", "Best assignment found by SA"]
costs = [greedy_cost, sa_result["final_cost"]]
bars = ax4.bar(method_names, costs, color=["lightblue", "lightcoral"], alpha=0.85)
ax4.set_ylabel("Total freight spend ($)")
ax4.set_title("Freight spend comparison")
ax4.tick_params(axis="x", labelrotation=10)
ax4.grid(True, axis="y", alpha=0.3)
for bar, cost in zip(bars, costs):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${cost:,.0f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

fig.tight_layout()
plt.show()

## Performance analysis

The following diagnostics summarize the run. The reported acceptance rate counts all proposed neighbors, including infeasible proposals, as part of the denominator; infeasible proposals are rejected. The convergence indicator is a descriptive window-based measure of variation in the recorded current-state objective. It is not a proof that the search has reached an optimum.

In [ ]:
def analyze_sa_performance(result, initial_temp, cooling_rate):
    """Print descriptive statistics for one Simulated Annealing run."""
    progress = result["progress_history"]
    acceptances = result["acceptance_history"]
    temperatures = result["temperature_history"]

    print("=" * 70)
    print("SIMULATED ANNEALING - PERFORMANCE ANALYSIS")
    print("=" * 70)
    print("\nBASIC STATISTICS:")
    print(f"   Iterations: {result['iterations']}")
    print(f"   Computation time: {result['computation_time']:.3f} seconds")
    print(f"   Best objective score: {result['best_objective']:.2f}")
    print(f"   Total freight spend: ${result['final_cost']:,.2f}")
    print(f"   Initial temperature: {initial_temp:.2f}")
    print(f"   Final temperature: {result['final_temperature']:.6f}")

    print("\nCURRENT-STATE OBJECTIVE DIAGNOSTICS:")
    if progress:
        best_recorded = min(progress)
        worst_recorded = max(progress)
        scale = max(abs(worst_recorded), np.finfo(float).eps)
        variation = (worst_recorded - best_recorded) / scale * 100
        print(f"   Best recorded current-state score: {best_recorded:.2f}")
        print(f"   Worst recorded current-state score: {worst_recorded:.2f}")
        print(f"   Relative range of recorded scores: {variation:.2f}%")

        window = min(100, len(progress))
        threshold = 0.01 * scale
        convergence_point = None
        if window > 0:
            for start in range(len(progress) - window + 1):
                recent = progress[start:start + window]
                if max(recent) - min(recent) < threshold:
                    convergence_point = start + window
                    break

        if convergence_point is None:
            print("   Window-based low-variation indicator: not reached")
        else:
            print(
                "   First window-based low-variation indicator: "
                f"iteration {convergence_point}"
            )
    else:
        print("   No iterations were performed.")

    print("\nACCEPTANCE STATISTICS:")
    if acceptances:
        midpoint = len(acceptances) // 2
        overall_rate = float(np.mean(acceptances))
        early_rate = float(np.mean(acceptances[:midpoint])) if midpoint else 0.0
        late_rate = float(np.mean(acceptances[midpoint:])) if midpoint < len(acceptances) else 0.0
        print(f"   Accepted proposals: {sum(acceptances)}/{len(acceptances)}")
        print(f"   Overall acceptance rate: {overall_rate * 100:.1f}%")
        print(f"   First-half acceptance rate: {early_rate * 100:.1f}%")
        print(f"   Second-half acceptance rate: {late_rate * 100:.1f}%")
        print(f"   Change between halves: {(early_rate - late_rate) * 100:.1f} percentage points")
    else:
        print("   No proposals were evaluated.")

    print("\nTEMPERATURE SCHEDULE:")
    print(f"   Cooling rate per iteration: {cooling_rate}")
    print(f"   Final temperature: {result['final_temperature']:.6f}")
    if temperatures:
        reduction = (1 - result["final_temperature"] / initial_temp) * 100
        print(f"   Temperature reduction: {reduction:.2f}%")


analyze_sa_performance(
    sa_result,
    initial_temp=1000,
    cooling_rate=0.995,
)

## Parameter sensitivity

Changing the initial temperature, cooling rate, or iteration limit changes how many moves are explored and how often worse feasible moves can be accepted. The following experiment retains the same problem data and initial assignment while varying those search parameters. Because SA is stochastic, results are outcomes of these runs rather than guarantees about future runs. Computation times are machine-dependent.

In [ ]:
def test_parameter_sensitivity(problem, initial_solution):
    """Run the same SA method under several temperature schedules."""
    test_configs = [
        {"initial_temp": 500, "cooling_rate": 0.99, "max_iterations": 3000},
        {"initial_temp": 1000, "cooling_rate": 0.995, "max_iterations": 5000},
        {"initial_temp": 2000, "cooling_rate": 0.998, "max_iterations": 7000},
        {"initial_temp": 1500, "cooling_rate": 0.992, "max_iterations": 4000},
    ]

    results = []
    for config_idx, config in enumerate(test_configs, start=1):
        print(f"\nTesting configuration {config_idx}: {config}")
        result = simulated_annealing(
            initial_solution,
            problem,
            initial_temp=config["initial_temp"],
            cooling_rate=config["cooling_rate"],
            max_iterations=config["max_iterations"],
        )
        acceptance_rate = (
            float(np.mean(result["acceptance_history"]))
            if result["acceptance_history"]
            else 0.0
        )
        results.append({
            "configuration": (
                f"T0={config['initial_temp']}, "
                f"cooling={config['cooling_rate']}, "
                f"max_iter={config['max_iterations']}"
            ),
            "final_cost": result["final_cost"],
            "best_objective": result["best_objective"],
            "computation_time": result["computation_time"],
            "iterations": result["iterations"],
            "acceptance_rate": acceptance_rate,
        })
        print(f"   Total freight spend: ${result['final_cost']:,.2f}")
        print(f"   Best objective score: {result['best_objective']:.2f}")
        print(f"   Computation time: {result['computation_time']:.3f} seconds")
        print(f"   Acceptance rate: {acceptance_rate * 100:.1f}%")

    return pd.DataFrame(results)


sensitivity_results = test_parameter_sensitivity(problem, initial_solution)
print("\nPARAMETER SENSITIVITY RESULTS")
print(sensitivity_results.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Sensitivity to Simulated Annealing Parameters", fontsize=16, fontweight="bold")
configuration_labels = [f"Configuration {idx}" for idx in range(1, len(sensitivity_results) + 1)]

ax1.bar(
    configuration_labels,
    sensitivity_results["final_cost"],
    color="lightcoral",
    alpha=0.85,
)
ax1.set_xlabel("Parameter configuration")
ax1.set_ylabel("Total freight spend ($)")
ax1.set_title("Freight spend by configuration")
ax1.tick_params(axis="x", labelrotation=25)
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(
    configuration_labels,
    sensitivity_results["computation_time"],
    color="lightgreen",
    alpha=0.85,
)
ax2.set_xlabel("Parameter configuration")
ax2.set_ylabel("Computation time (seconds)")
ax2.set_title("Computation time by configuration")
ax2.tick_params(axis="x", labelrotation=25)
ax2.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

## Interpretation and limitations

Simulated Annealing provides a structured way to search a discrete assignment space without enumerating every feasible assignment. Its temperature-dependent acceptance rule can cross objective-increasing moves, while the best-so-far record preserves the strongest solution encountered. In this notebook, the neighborhood changes one lane's carrier, and carrier capacities determine whether a proposed assignment can be considered.

The returned assignment is the best feasible assignment observed during the run, not a certificate of global optimality. Results can depend on the greedy starting point, the random sequence, and the temperature parameters. The sensitivity experiment illustrates the effect of parameter choices on this instance; it does not establish that one schedule is best for other instances.

This method is useful when a practical search over a discrete assignment space is desired and a guaranteed optimum is not required. Larger or richer applications may add business constraints and extend the state representation while retaining the same search principle.

In [ ]:
print("=" * 70)
print("SIMULATED ANNEALING - FINAL ASSESSMENT")
print("=" * 70)

print("\nBEST FEASIBLE ASSIGNMENT FOUND:")
for lane_idx in np.argsort(-problem["demand"]):
    carrier_idx = sa_result["best_assignments"][lane_idx]
    print(f"   {problem['lanes'][lane_idx]}: {problem['carriers'][carrier_idx]}")

feasible, carrier_utilization = is_feasible(sa_result["best_assignments"], problem)
assert feasible
assert np.isclose(
    calculate_total_cost(sa_result["best_assignments"], problem),
    sa_result["final_cost"],
)
assert np.isclose(
    calculate_objective(sa_result["best_assignments"], problem),
    sa_result["best_objective"],
)

print("\nRESULT METRICS:")
print(f"   Weighted objective score: {sa_result['best_objective']:.2f}")
print(f"   Total freight spend: ${sa_result['final_cost']:,.2f}")
print(f"   Iterations completed: {sa_result['iterations']}")
print(f"   Computation time: {sa_result['computation_time']:.3f} seconds")
print(f"   Feasible assignment: {feasible}")
print("   Carrier utilization and capacity:")
for carrier_idx, carrier in enumerate(problem["carriers"]):
    print(
        f"      {carrier}: {carrier_utilization[carrier_idx]:g} "
        f"of {problem['capacity_limits'][carrier_idx]:g} demand units"
    )

print("\nCOMPARISON WITH THE GREEDY INITIAL ASSIGNMENT:")
if np.isclose(greedy_cost, 0.0):
    print("   Percentage change in freight spend is undefined for a zero baseline.")
else:
    spend_change = (sa_result["final_cost"] - greedy_cost) / greedy_cost * 100
    print(f"   Greedy freight spend: ${greedy_cost:,.2f}")
    print(f"   Simulated Annealing freight spend: ${sa_result['final_cost']:,.2f}")
    print(f"   Freight-spend change relative to greedy: {spend_change:.2f}%")
print(f"   Greedy objective score: {greedy_objective:.2f}")
print(f"   Simulated Annealing best objective score: {sa_result['best_objective']:.2f}")

print("\nMETHOD NOTES:")
print("   - Worse feasible moves may be accepted according to the current temperature.")
print("   - Infeasible proposals are rejected, and the temperature still decreases.")
print("   - The best feasible state encountered is reported; global optimality is not guaranteed.")
print("   - The weighted objective score and actual freight spend are distinct quantities.")

print("\n" + "=" * 70)
print("ASSESSMENT COMPLETE")
print("=" * 70)